# Session 20 Lab: Capstone Project Starter
**Course:** Noob to AI Expert | **Track:** Expert

This notebook gives you a production-ready starter template for any of the 5 capstone tracks. Choose your track and build on this foundation.

In [ ]:
!pip install anthropic openai chromadb streamlit fastapi uvicorn python-dotenv -q
print('✅ Ready')

In [ ]:
# ============================================================
# CAPSTONE STARTER TEMPLATE
# Replace the placeholders below to build your chosen track.
# ============================================================

import anthropic, json, hashlib, uuid, time, logging, re
from collections import defaultdict
import os

os.environ.setdefault('ANTHROPIC_API_KEY', 'your-key-here')
client = anthropic.Anthropic()

# -------- CONFIGURE YOUR APP --------
APP_NAME = 'My Capstone AI App'        # Change this
SYSTEM_PROMPT = """You are a helpful AI assistant.
# TODO: Customize with your persona, constraints, and format instructions
"""

# -------- RESPONSE CACHE --------
_cache = {}
def cache_key(messages, system): return hashlib.md5(json.dumps({'m': messages, 's': system}).encode()).hexdigest()
def cached_call(messages, system=SYSTEM_PROMPT, max_tokens=512):
    key = cache_key(messages, system)
    if key in _cache: return _cache[key]
    resp = client.messages.create(model='claude-haiku-4-5-20251001', max_tokens=max_tokens, system=system, messages=messages)
    result = resp.content[0].text
    _cache[key] = result
    return result

# -------- RATE LIMITER --------
_windows = defaultdict(list)
def is_allowed(user_id, rpm=10):
    now = time.time()
    _windows[user_id] = [t for t in _windows[user_id] if now-t < 60]
    if len(_windows[user_id]) >= rpm: return False
    _windows[user_id].append(now); return True

# -------- EVAL SUITE --------
GOLDEN_DATASET = [
    # TODO: Add 20+ test cases for your specific app
    {'input': 'Hello', 'expected_contains': [], 'should_not_contain': []},
]

def run_eval(dataset=GOLDEN_DATASET):
    results = []
    for case in dataset:
        output = cached_call([{'role': 'user', 'content': case['input']}])
        passed = all(keyword in output for keyword in case.get('expected_contains', []))
        safe = not any(bad in output for bad in case.get('should_not_contain', []))
        results.append({'input': case['input'], 'output': output, 'passed': passed and safe})
    pass_rate = sum(1 for r in results if r['passed']) / len(results)
    return {'pass_rate': pass_rate, 'total': len(results)}

print(f'{APP_NAME} — Starter loaded')
print('Next steps:')
print('  1. Customize SYSTEM_PROMPT for your track')
print('  2. Add 20+ test cases to GOLDEN_DATASET')
print('  3. Add RAG or tool calling as required')
print('  4. Build Streamlit UI in app.py')
print('  5. Deploy to Streamlit Cloud')

In [ ]:
# Quick smoke test
result = cached_call([{'role': 'user', 'content': 'Hello! What can you help me with?'}])
print('Response:', result[:300])

eval_results = run_eval()
print(f'\nEval: {eval_results["pass_rate"]:.0%} pass rate ({eval_results["total"]} cases)')

## Next: Choose Your Track

**Track A — AI Tutor**: Add RAG over study materials. Add quiz generation tools.

**Track B — Resume Reviewer**: Parse PDF, compare to job description, output JSON with scores.

**Track C — Knowledge Base Bot**: Ingest documents with ChromaDB. Add citation tracking.

**Track D — Meeting Assistant**: Add Whisper transcription. Extract action items as structured JSON.

**Track E — AI Coding Assistant**: Add code execution sandbox. Generate and run tests.

See the capstone checklist in Session 20 for the full requirements.